
# HMI PFSS solutions

Calculating a PFSS solution from a HMI synoptic map.

This example shows how to calculate a PFSS solution from a HMI synoptic map.
There are a couple of important things that this example shows:

- HMI maps have non-standard metadata, so this needs to be fixed
- HMI synoptic maps are very big (1440 x 3600), so need to be downsampled
  in order to calculate the PFSS solution in a reasonable time.


In [1]:
import os

import matplotlib.pyplot as plt

import astropy.units as u

import sunpy.map
from sunpy.net import Fido
from sunpy.net import attrs as a

from sunkit_magex import pfss

/Users/xwu/opt/miniconda3/envs/sunpy_7/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Set up the search.

The synoptic maps are labelled by Carrington rotation number instead of time.



In [2]:
series = a.jsoc.Series('hmi.synoptic_mr_polfil_720s')
crot = a.jsoc.PrimeKey('CAR_ROT', 2210)

Do the search.

If you use this code, please replace this email address
with your own one, registered here:
http://jsoc.stanford.edu/ajax/register_email.html



In [ ]:
# result = Fido.search(series, crot, a.jsoc.Notify('xwu0011@gmail.com'))
# files = Fido.fetch(result)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Read in a file. This will read in the first file downloaded to a sunpy Map
object.



In [41]:
hmi_file='/Users/xwu/Library/CloudStorage/OneDrive-UniversityCollegeLondon/MSSL/Project/Database/HMI/hmi_m_45s_2022_02_28_20_08_15_tai_magnetogram.fits'
hmi_map = sunpy.map.Map(hmi_file) #files[0]

Since this map is far to big to calculate a PFSS solution quickly, lets
resample it down to a smaller size.



In [46]:
print('Old shape: ', hmi_map.data.shape)
hmi_map = hmi_map.resample([360, 360] * u.pix)
print('New shape: ', hmi_map.data.shape)

Old shape:  (4096, 4096)
New shape:  (360, 360)


Now calculate the PFSS solution



In [47]:
hpc_coords = sunpy.map.all_coordinates_from_map(hmi_map)
mask = ~sunpy.map.coordinate_is_on_solar_disk(hpc_coords)
hmiMap = sunpy.map.Map(hmi_map.data, hmi_map.meta, mask=mask) #

In [44]:
nrho = 35
rss = 2.5
pfss_in = pfss.Input(hmiMap, nrho, rss)
pfss_out = pfss.pfss(pfss_in)

ValueError: At least one value in the input is NaN or non-finite. The input must consist solely of finite values.

Using the Output object we can plot the source surface field, and the
polarity inversion line.



In [ ]:
ss_br = pfss_out.source_surface_br

fig = plt.figure()
ax = fig.add_subplot(projection=ss_br)

ss_br.plot(axes=ax)
# Plot the polarity inversion line
ax.plot_coord(pfss_out.source_surface_pils[0])
plt.colorbar()
ax.set_title('Source surface magnetic field')

plt.show()

#### Error Diagnostic

In [ ]:
# === JSOC 诊断脚本（版本含你的邮箱） ===
# pip install requests certifi drms

import os, sys, ssl, json, traceback
from typing import Tuple, Optional

EMAIL = "xwu0011@gmail.com"
URL_TMPL = "https://jsoc.stanford.edu/cgi-bin/ajax/checkAddress?address={email}&checkonly=1"

def print_header(title):
    print("\n" + "="*80)
    print(title)
    print("="*80)

def get_env_proxies():
    keys = ["HTTP_PROXY","HTTPS_PROXY","http_proxy","https_proxy","ALL_PROXY","all_proxy","NO_PROXY","no_proxy"]
    return {k: os.environ.get(k) for k in keys if os.environ.get(k) is not None}

def sniff_text_kind(text: str) -> str:
    low = text.strip().lower()
    if not low:
        return "empty"
    if low.startswith("<") or "html" in low[:200]:
        return "html"
    try:
        json.loads(text)
        return "json"
    except json.JSONDecodeError:
        return "plain"

def try_requests(email: str, verify: Optional[str|bool]) -> Tuple[bool, Optional[str], Optional[str]]:
    try:
        import requests
    except Exception as e:
        return False, None, f"requests 导入失败: {e}"

    url = URL_TMPL.format(email=email)
    try:
        r = requests.get(url, timeout=25, verify=verify)
        txt = (r.text or "")[:1024]
        return True, txt, None
    except Exception as e:
        return False, None, f"{e.__class__.__name__}: {e}"

def parse_jsoc_json(text: str) -> dict:
    try:
        return json.loads(text)
    except Exception:
        return {}

def try_drms_check(email: str, unverified_ssl: bool=False) -> Tuple[Optional[bool], Optional[str]]:
    try:
        import drms
    except Exception as e:
        return None, f"drms 导入失败: {e}"

    if unverified_ssl:
        ssl._create_default_https_context = ssl._create_unverified_context

    try:
        c = drms.Client()
        ok = c.check_email(email)
        return bool(ok), None
    except Exception as e:
        return None, f"{e.__class__.__name__}: {e}"

def main():
    print_header("环境信息")
    print(f"Python: {sys.version.split()[0]}  | Executable: {sys.executable}")
    try:
        import ssl, certifi
        print(f"OpenSSL: {ssl.OPENSSL_VERSION}")
        print(f"certifi CA file: {certifi.where()}")
    except Exception as e:
        print(f"certifi/ssl 信息获取失败: {e}")

    proxies = get_env_proxies()
    print("代理环境变量（如有）:")
    if proxies:
        for k,v in proxies.items():
            print(f"  {k} = {v}")
    else:
        print("  （未发现代理相关环境变量）")

    print_header("Step 1: 直连（系统证书）访问 JSOC checkAddress")
    ok1, text1, err1 = try_requests(EMAIL, verify=True)
    if ok1:
        kind1 = sniff_text_kind(text1)
        print(f"直连成功，返回内容类型：{kind1}")
        if kind1 == "json":
            data = parse_jsoc_json(text1)
            print(f"返回 JSON: {data}")
        else:
            print(f"返回前 1KB 文本（截断）：\n{text1}")
    else:
        print(f"直连失败：{err1}")

    print_header("Step 2: 使用 certifi CA 访问")
    cafile = None
    try:
        import certifi
        cafile = certifi.where()
    except Exception as e:
        print(f"无法使用 certifi：{e}")

    if cafile:
        ok2, text2, err2 = try_requests(EMAIL, verify=cafile)
        if ok2:
            kind2 = sniff_text_kind(text2)
            print(f"certifi 访问成功，返回内容类型：{kind2}")
            if kind2 == "json":
                data2 = parse_jsoc_json(text2)
                print(f"返回 JSON: {data2}")
            else:
                print(f"返回前 1KB 文本（截断）：\n{text2}")
        else:
            print(f"certifi 访问失败：{err2}")
    else:
        ok2, text2, err2 = False, None, "无 certifi"

    print_header("Step 3:（仅诊断）忽略证书校验访问")
    ok3, text3, err3 = try_requests(EMAIL, verify=False)
    if ok3:
        kind3 = sniff_text_kind(text3)
        print(f"忽略校验访问成功，返回内容类型：{kind3}")
        if kind3 == "json":
            data3 = parse_jsoc_json(text3)
            print(f"返回 JSON: {data3}")
        else:
            print(f"返回前 1KB 文本（截断）：\n{text3}")
    else:
        print(f"忽略校验访问仍失败：{err3}")

    print_header("Step 4: 调用 drms.Client().check_email()（正常证书）")
    ok4, err4 = try_drms_check(EMAIL, unverified_ssl=False)
    if ok4 is True:
        print("✅ drms.check_email 结果：True（邮箱已注册）")
    elif ok4 is False:
        print("⚠️ drms.check_email 结果：False（访问正常但未注册）")
    else:
        print(f"❌ drms.check_email 调用失败：{err4}")

    print_header("Step 5:（仅诊断）drms 忽略证书校验后再试")
    ok5, err5 = try_drms_check(EMAIL, unverified_ssl=True)
    if ok5 is True:
        print("✅ drms.check_email(unverified)：True（邮箱已注册，证书问题导致失败）")
    elif ok5 is False:
        print("⚠️ drms.check_email(unverified)：False（访问通但邮箱未注册）")
    else:
        print(f"❌ drms.check_email(unverified) 调用失败：{err5}")

    # -------- 结论 --------
    print_header("结论摘要")
    def saw_json(txt): return txt and sniff_text_kind(txt) == "json"
    if not ok1 and "SSLError" in (err1 or ""):
        print("🔎 结论：HTTPS 证书验证失败。")
        if ok2:
            print("👉 用 certifi 能成功，建议设置：\n  export SSL_CERT_FILE=$(python -m certifi)\n  export REQUESTS_CA_BUNDLE=$(python -m certifi)")
        elif ok3:
            print("👉 忽略校验能成功 → 说明是证书信任链问题（常见于校园/公司代理）")
        else:
            print("👉 证书和网络可能都有问题。")
    elif saw_json(text1) or saw_json(text2) or saw_json(text3):
        data = None
        for t in (text1, text2, text3):
            if saw_json(t):
                data = parse_jsoc_json(t); break
        status = int(data.get("status", -1)) if data else -1
        if status == 2:
            print("✅ 邮箱已注册并通过验证。若仍出错，说明是 Python 层证书问题。")
        elif status in (0,1):
            print("⚠️ 邮箱未注册或校验未通过，请前往 JSOC 注册并使用相同邮箱。")
        else:
            print("❓ 收到未知 JSON，可能是服务端响应异常。")
    else:
        print("❌ 无法获取 JSON，且所有访问失败 → 网络、证书或代理问题未解决。")

if __name__ == "__main__":
    main()



环境信息
Python: 3.13.7  | Executable: /Users/xwu/opt/miniconda3/envs/sunpy_7/bin/python
OpenSSL: OpenSSL 3.5.4 30 Sep 2025
certifi CA file: /Users/xwu/opt/miniconda3/envs/sunpy_7/lib/python3.13/site-packages/certifi/cacert.pem
代理环境变量（如有）:
  （未发现代理相关环境变量）

Step 1: 直连（系统证书）访问 JSOC checkAddress
直连失败：SSLError: HTTPSConnectionPool(host='jsoc.stanford.edu', port=443): Max retries exceeded with url: /cgi-bin/ajax/checkAddress?address=xwu0011@gmail.com&checkonly=1 (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_HANDSHAKE_FAILURE] ssl/tls alert handshake failure (_ssl.c:1032)')))

Step 2: 使用 certifi CA 访问
certifi 访问失败：SSLError: HTTPSConnectionPool(host='jsoc.stanford.edu', port=443): Max retries exceeded with url: /cgi-bin/ajax/checkAddress?address=xwu0011@gmail.com&checkonly=1 (Caused by SSLError(SSLError(1, '[SSL: SSLV3_ALERT_HANDSHAKE_FAILURE] ssl/tls alert handshake failure (_ssl.c:1032)')))

Step 3:（仅诊断）忽略证书校验访问
忽略校验访问仍失败：SSLError: HTTPSConnectionPool(host='jsoc.stanford.edu', port=443): 